## Get all the imports

In [1]:
import pytorch_lightning as pl
import torch
from torchvision import transforms
from torchvision.datasets import CIFAR10
from model import MyModel
from datacull.data import DCDataset
from datacull.methods.MetriQ import MetriQDataLoader
from datacull.logger import DCLogger
import numpy as np

## Defining some global variables

In [2]:
num_epochs = 20
batch_size = 256
pruning_rate = 0.8

## Create the data module class (CIFAR10 for simplicity)
- Since MetriQ is a static pruning technique, we need to first identify the class ratios.
- For that, we build a normal pytorch lightning data module.
- The only difference is that the **pytorch dataset** is going to be **wrapped by the DCDataset** to enable indexing.

In [3]:
class PrePruningDataModule(pl.LightningDataModule):
    def __init__(self, batch_size):
        self.batch_size = batch_size
        # These are the transforms to be used if you want to train a model from scratch on the current dataset
        self.train_transform = transforms.Compose([
            transforms.RandomCrop(32, 4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])
        self.val_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])
        super().__init__()
    
    
    def prepare_data(self, stage="None"):
        # This is the only line where we differ from normal code.
        # We wrap the CIFAR10 dataset (or your own dataset) with DPDataset to enable indexing
        self.train_set = DCDataset(CIFAR10(root='data/', train=True, download=True, transform=self.train_transform))
        self.val_set = DCDataset(CIFAR10(root='data/', train=False, download=True, transform=self.val_transform))
    
    
    def setup(self, stage: str):
        self.train_data_loader = torch.utils.data.DataLoader(self.train_set, batch_size=self.batch_size, shuffle=True, num_workers=1)
        self.val_data_loader = torch.utils.data.DataLoader(self.val_set, batch_size=self.batch_size, shuffle=False, num_workers=1)
    
    
    def train_dataloader(self):
        return self.train_data_loader
    
    
    def val_dataloader(self):
        return self.val_data_loader

## Define a model for training
- We will use its training trajectory to find sample importance
- The logger object is used to log the metrics every iteration
- Check line 41 in model.py

In [4]:
model = MyModel(logger_object=None)

## Create the data module object and train the model

In [5]:
data_module = PrePruningDataModule(batch_size=batch_size)
trainer = pl.Trainer(accelerator="gpu", devices=1, max_epochs=num_epochs, deterministic=True, enable_model_summary=True, num_sanity_val_steps=0)
# trainer.fit(model, data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\ProgramData\anaconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


## Find the class-wise accuracies

In [6]:
class_wise_acc = np.random.uniform(0, 1, 10)

## Next, create a data module that will be used after pruning is done
- To make lives simpler, we will inherit the PrePruningDataModule to avoid multiple definitions of transforms, etc.
- This is the class where we use the MetriQDataLoader to apply MetriQ pruning logic
- Since MetriQ is a static data pruning technique, the resample function ensures we sample once and then only shuffle the data every epoch

In [7]:
class PostPruningDataModule(PrePruningDataModule):
    def __init__(self, pruning_rate, batch_size):
        self.pruning_rate = 1 - pruning_rate
        super().__init__(batch_size)
    
    def setup(self, stage: str):
        # Use the MetriQDataLoader to apply MetriQ sampling strategy
        self.train_data_loader = MetriQDataLoader(dataset=self.train_set, pruning_rate=self.pruning_rate, class_wise_acc=class_wise_acc, batch_size=self.batch_size, num_workers=1)
        self.val_data_loader = torch.utils.data.DataLoader(self.val_set, batch_size=self.batch_size, shuffle=False, num_workers=1)
    
    def train_dataloader(self):
        # Resample the training data loader based on sample importance
        self.train_data_loader.resample(None)
        return self.train_data_loader


## Finally, train a new model on a data subset
- The subset is calculated using the sample_importance

In [8]:
data_module = PostPruningDataModule(pruning_rate=pruning_rate, batch_size=batch_size)
model = MyModel()
# reload_dataloaders_every_n_epochs=True is important to ensure that the dataloader is reloaded every epoch to reflect the new sampling.
# For static methods, this only shuffles the data once the subset is selected, but for dynamic methods, this is necessary to update the sampling based on new importance scores.
# This is a design choice to unify both static and dynamic methods under the same flag.
trainer = pl.Trainer(accelerator="gpu", devices=1, max_epochs=num_epochs, deterministic=True, enable_model_summary=False, num_sanity_val_steps=0, reload_dataloaders_every_n_epochs=True)
trainer.fit(model, data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\ProgramData\anaconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:428: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.
c:\ProgramData\anaconda3\Lib\site-packages\pytorch_lightning\loops\fit_loop.py:310: The number of training batches (40) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
c:\ProgramData\anaconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:428: Consider setting `persistent_workers=Tru

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.
